In [1]:
print("hi")

hi


Multi Head Attention

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:

sequence_length = 4
batch_size = 1
input_dim = 512
d_model = 512
x = torch.randn( (batch_size, sequence_length, input_dim) )

In [4]:
x.shape

torch.Size([1, 4, 512])

In [5]:

qkv_layer = nn.Linear(input_dim , 3 * d_model)

In [6]:

qkv = qkv_layer(x)

In [7]:
qkv.shape

torch.Size([1, 4, 1536])

In [8]:
num_heads = 8
head_dim = d_model // num_heads
qkv = qkv.reshape(batch_size, sequence_length, num_heads, 3 * head_dim)

In [9]:

qkv.shape

torch.Size([1, 4, 8, 192])

In [10]:

qkv = qkv.permute(0, 2, 1, 3) # [batch_size, num_heads, sequence_length, 3*head_dim]
qkv.shape

torch.Size([1, 8, 4, 192])

In [11]:
q, k, v = qkv.chunk(3, dim=-1)
q.shape, k.shape, v.shape

(torch.Size([1, 8, 4, 64]),
 torch.Size([1, 8, 4, 64]),
 torch.Size([1, 8, 4, 64]))

In [13]:
import math
d_k = q.size()[-1]
scaled = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
scaled.shape

torch.Size([1, 8, 4, 4])

In [14]:
#don't do that 
k.T.shape

C:\Users\indroneel\AppData\Local\Temp\ipykernel_17924\3821867643.py:2: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4419.)
  k.T.shape


torch.Size([64, 4, 8, 1])

In [18]:
k.transpose(-1, -2).shape

torch.Size([1, 8, 64, 4])

In [22]:

mask = torch.full(scaled.size() , float('-inf'))
mask = torch.triu(mask, diagonal=1)
mask[0][0]

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

In [23]:
(scaled + mask)[0][0]

tensor([[ 0.0616,    -inf,    -inf,    -inf],
        [ 0.0380, -0.1413,    -inf,    -inf],
        [-0.0303, -0.0609,  0.1343,    -inf],
        [ 0.9031,  0.5586,  0.7254, -0.0656]], grad_fn=<SelectBackward0>)

In [24]:
scaled += mask

In [26]:
scaled[0][0]

tensor([[ 0.0616,    -inf,    -inf,    -inf],
        [ 0.0380, -0.1413,    -inf,    -inf],
        [-0.0303, -0.0609,  0.1343,    -inf],
        [ 0.9031,  0.5586,  0.7254, -0.0656]], grad_fn=<SelectBackward0>)

In [27]:
attention = F.softmax(scaled, dim=-1)

In [28]:

attention.shape

torch.Size([1, 8, 4, 4])

In [29]:
attention[0][0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5447, 0.4553, 0.0000, 0.0000],
        [0.3176, 0.3080, 0.3744, 0.0000],
        [0.3418, 0.2422, 0.2862, 0.1298]], grad_fn=<SelectBackward0>)

In [30]:
values = torch.matmul(attention, v)
values.shape

torch.Size([1, 8, 4, 64])

Function

In [31]:

import math

def scaled_dot_product(q, k, v, mask=None):
    d_k = q.size()[-1]
    scaled = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scaled += mask
    attention = F.softmax(scaled, dim=-1)
    values = torch.matmul(attention, v)
    return values, attention

In [36]:

values, attention = scaled_dot_product(q, k, v, mask=None)

In [33]:
attention.shape

torch.Size([1, 8, 4, 4])

In [37]:
attention[0][0]

tensor([[0.2290, 0.1906, 0.3445, 0.2359],
        [0.2729, 0.2281, 0.1733, 0.3257],
        [0.2137, 0.2073, 0.2520, 0.3270],
        [0.3418, 0.2422, 0.2862, 0.1298]], grad_fn=<SelectBackward0>)

In [34]:
values.shape

torch.Size([1, 8, 4, 64])

In [38]:
values.size()

torch.Size([1, 8, 4, 64])

In [39]:
values = values.reshape(batch_size, sequence_length, num_heads * head_dim)
values.size()

torch.Size([1, 4, 512])

In [40]:
linear_layer = nn.Linear(d_model, d_model)

In [41]:
out = linear_layer(values)

In [42]:
out.shape

torch.Size([1, 4, 512])

In [44]:

out

tensor([[[-6.4031e-02,  2.1765e-02,  1.4313e-01,  ...,  3.4716e-01,
           2.5676e-01,  2.5020e-04],
         [-8.9600e-02, -6.7086e-02, -1.0264e-01,  ...,  1.6179e-01,
          -2.6627e-01,  1.7159e-01],
         [-8.7161e-03,  1.9296e-01, -2.1975e-01,  ...,  2.6841e-02,
          -1.2485e-01, -5.2845e-02],
         [-3.1851e-01,  4.8357e-01,  3.3053e-02,  ...,  1.7342e-01,
          -1.7141e-01, -9.6802e-02]]], grad_fn=<ViewBackward0>)

Class

In [45]:
import torch
import torch.nn as nn
import math

def scaled_dot_product(q, k, v, mask=None):
    d_k = q.size()[-1]
    scaled = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scaled += mask
    attention = F.softmax(scaled, dim=-1)
    values = torch.matmul(attention, v)
    return values, attention

class MultiheadAttention(nn.Module):

    def __init__(self, input_dim, d_model, num_heads):
        super().__init__()
        self.input_dim = input_dim
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_layer = nn.Linear(input_dim , 3 * d_model)
        self.linear_layer = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, sequence_length, input_dim = x.size()
        print(f"x.size(): {x.size()}")
        qkv = self.qkv_layer(x)
        print(f"qkv.size(): {qkv.size()}")
        qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3 * self.head_dim)
        print(f"qkv.size(): {qkv.size()}")
        qkv = qkv.permute(0, 2, 1, 3)
        print(f"qkv.size(): {qkv.size()}")
        q, k, v = qkv.chunk(3, dim=-1)
        print(f"q size: {q.size()}, k size: {k.size()}, v size: {v.size()}, ")
        values, attention = scaled_dot_product(q, k, v, mask)
        print(f"values.size(): {values.size()}, attention.size:{ attention.size()} ")
        values = values.reshape(batch_size, sequence_length, self.num_heads * self.head_dim)
        print(f"values.size(): {values.size()}")
        out = self.linear_layer(values)
        print(f"out.size(): {out.size()}")
        return out

In [46]:
input_dim = 1024
d_model = 512
num_heads = 8

batch_size = 30
sequence_length = 5
x = torch.randn( (batch_size, sequence_length, input_dim) )

model = MultiheadAttention(input_dim, d_model, num_heads)
out = model.forward(x)

x.size(): torch.Size([30, 5, 1024])
qkv.size(): torch.Size([30, 5, 1536])
qkv.size(): torch.Size([30, 5, 8, 192])
qkv.size(): torch.Size([30, 8, 5, 192])
q size: torch.Size([30, 8, 5, 64]), k size: torch.Size([30, 8, 5, 64]), v size: torch.Size([30, 8, 5, 64]), 
values.size(): torch.Size([30, 8, 5, 64]), attention.size:torch.Size([30, 8, 5, 5]) 
values.size(): torch.Size([30, 5, 512])
out.size(): torch.Size([30, 5, 512])
